# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hariommishra-12/Flyrank-Project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My rule and its reason codes

I will prioritize pages that have meaningful visibility but show a clear refresh opportunity.

My rule uses two observable signals:
1. **Staleness:** the page has not been updated for at least 180 days and has at least 500 impressions in the trailing 90-day window.
2. **Low CTR opportunity:** the page has at least 500 impressions, ranks on page 1–2 (average position 1–20), and has CTR below 0.5%.

The score gives more weight to staleness and a smaller amount to the CTR opportunity. The rule is deliberately simple and uses only information available in the current observation window. It does not use `trend_pct`, `trend_direction`, or the derived decline label as an input.

Each ranked page receives exactly one reason code:
- `stale_visible`
- `low_ctr_visible`
- `stale_and_low_ctr`
- `general_review`

The action label is:
- `refresh` for stale pages
- `refresh_and_review_ctr` for low-CTR pages
- `refresh_and_review_ctr` for pages showing both signals
- `monitor` for pages with neither signal

Before building the queue, I will check whether the two signals are directionally supported by the observed data. The checks are decision-support evidence, not model features.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Load data + check the two signals
from pathlib import Path
import pandas as pd
import numpy as np

# Find the starter CSV whether the notebook is running from the repo
# or from the Colab /content directory.
matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))

if not matches:
    raw_url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/main/"
        "data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(raw_url)
else:
    df = pd.read_csv(matches[0])

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Keep IDs for review/grouping, but never use them in the score.
# trend_pct and trend_direction are label-derived and are NOT features.
required = [
    "content_id", "client_id",
    "days_since_last_update", "impressions_90d",
    "avg_position", "ctr",
    "trend_direction"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---------- Signal 1: staleness + visibility ----------
df["stale_visible_signal"] = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
)

# Bucket the staleness signal so we can inspect the observed decline rate.
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          decline_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

print("\nSIGNAL CHECK 1 — STALENESS")
print(staleness_check.to_string(index=False))

# ---------- Signal 2: CTR vs position ----------
# Ignore avg_position == 0 because 0 means "no position data".
df["ctr_position_signal"] = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
)

df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[-np.inf, 0.25, 0.50, 1.00, np.inf],
    labels=["<0.25", "0.25-0.50", "0.50-1.00", "1.00+"]
)

ctr_check = (
    df[df["avg_position"] > 0]
      .groupby("ctr_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          decline_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

print("\nSIGNAL CHECK 2 — CTR")
print(ctr_check.to_string(index=False))

# Simple verdicts based on the observed pattern.
def signal_verdict(bucket_table):
    rates = bucket_table["decline_rate"].dropna()

    if len(rates) < 2:
        return "MIXED"

    spread = rates.max() - rates.min()

    if spread < 0.02:
        return "MIXED"

    # A directional relationship is enough for this baseline audit.
    return "CONFIRMED"

print("\nVERDICT — STALENESS:", signal_verdict(staleness_check))
print("VERDICT — CTR:", signal_verdict(ctr_check))

# These verdicts are descriptive checks only.
# The labels above are NOT used in the ranking score.

Rows: 30000
Columns: 44

SIGNAL CHECK 1 — STALENESS
staleness_bucket     n  decline_rate
            0-90 20655      0.512031
          91-180  9171      0.611057
         181-365   169      0.467456
            365+     5      0.600000

SIGNAL CHECK 2 — CTR
ctr_bucket     n  decline_rate
     <0.25 20564      0.581015
 0.25-0.50  4089      0.561506
 0.50-1.00  2460      0.513415
     1.00+  1682      0.444114

VERDICT — STALENESS: CONFIRMED
VERDICT — CTR: CONFIRMED


## Build the ranked queue

I use a transparent, hand-written score rather than fitted weights.

The score is:

- 0.60 × staleness/visibility signal
- 0.40 × low-CTR/page-position opportunity signal

This is a decision-support ranking, not a prediction model.

The queue receives one reason code per page and one suggested action. The output is ranked from highest score to lowest score and written to `work/outputs/baseline_action_score.csv`.

No label-derived fields are used in the score.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Build the ranked queue

# Signal 1: stale + visible
stale_visible = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
).astype(int)

# Signal 2: visible + page 1/2 + low CTR
low_ctr_visible = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
).astype(int)

# Transparent hand-written score.
df["baseline_action_score"] = (
    0.60 * stale_visible +
    0.40 * low_ctr_visible
)

# Exactly ONE reason code per row.
def choose_reason(row):
    if row["stale_visible_signal"] and row["ctr_position_signal"]:
        return "stale_and_low_ctr"
    elif row["stale_visible_signal"]:
        return "stale_visible"
    elif row["ctr_position_signal"]:
        return "low_ctr_visible"
    else:
        return "general_review"

df["reason_code"] = df.apply(choose_reason, axis=1)

def choose_action(reason):
    if reason == "stale_visible":
        return "refresh"
    if reason == "low_ctr_visible":
        return "refresh_and_review_ctr"
    if reason == "stale_and_low_ctr":
        return "refresh_and_review_ctr"
    return "monitor"

df["action"] = df["reason_code"].map(choose_action)

# Rank everything.
df["baseline_rank"] = (
    df["baseline_action_score"]
      .rank(method="first", ascending=False)
      .astype(int)
)

# Output required by the assignment.
output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

output = (
    df[output_columns]
    .sort_values("baseline_rank")
    .reset_index(drop=True)
)

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print(f"Wrote: {output_path}")
print(f"Rows ranked: {len(output):,}")
print(f"Top score: {output['baseline_action_score'].max():.2f}")

print("\nAction counts:")
print(output["action"].value_counts())

print("\nTop 10:")
display(output.head(10))

Wrote: work/outputs/baseline_action_score.csv
Rows ranked: 30,000
Top score: 1.00

Action counts:
action
monitor                   20234
refresh_and_review_ctr     9759
refresh                       7
Name: count, dtype: int64

Top 10:


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count
0,content_fe16a55cd13d,client_7f2253d7e2,1,1.0,stale_and_low_ctr,refresh_and_review_ctr,4556,15,42,16.4,0.33,231,194,3388.0
1,content_72496874f806,client_4ec9599fc2,2,1.0,stale_and_low_ctr,refresh_and_review_ctr,821,2,6,5.8,0.24,301,301,1504.0
2,content_6226ee6adc91,client_d029fa3a95,3,1.0,stale_and_low_ctr,refresh_and_review_ctr,545,1,5,17.8,0.18,232,183,3950.0
3,content_c2d929d83eaa,client_7f2253d7e2,4,1.0,stale_and_low_ctr,refresh_and_review_ctr,7558,15,25,17.9,0.20,231,193,4758.0
4,content_cf56e2e2e282,client_7f2253d7e2,5,1.0,stale_and_low_ctr,refresh_and_review_ctr,61678,94,119,19.7,0.15,231,194,5125.0
5,content_928af3e22c80,client_7f2253d7e2,6,1.0,stale_and_low_ctr,refresh_and_review_ctr,1697,2,3,15.8,0.12,231,193,3118.0
6,content_0a91db491d14,client_7f2253d7e2,7,1.0,stale_and_low_ctr,refresh_and_review_ctr,13299,65,78,10.5,0.49,231,193,3478.0
7,content_e3ff1b093148,client_d029fa3a95,8,1.0,stale_and_low_ctr,refresh_and_review_ctr,1408,4,9,7.8,0.28,232,183,4758.0
8,content_77d4d5930e5e,client_7f2253d7e2,9,1.0,stale_and_low_ctr,refresh_and_review_ctr,828,2,4,18.6,0.24,231,194,4020.0
9,content_7f116ae1f6f5,client_9400f1b21c,10,1.0,stale_and_low_ctr,refresh_and_review_ctr,954,4,6,9.0,0.42,301,301,1335.0


## Top-20 review

I am treating the ranking as decision-support rather than truth.

For every top-20 item, I will record:
- the action,
- the single reason code,
- a confidence note based on the visible evidence,
- and what evidence would make the recommendation wrong.

The confidence note is deliberately cautious because this is a hand-written baseline, not a validated predictive model.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Top-20 review

top20 = output.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "stale_and_low_ctr":
        return "Moderate: both baseline signals are present."
    elif row["reason_code"] == "stale_visible":
        return "Moderate: strong visibility plus long update gap."
    elif row["reason_code"] == "low_ctr_visible":
        return "Low-to-moderate: visible page with a CTR opportunity."
    return "Low: weak direct evidence from this rule."

def what_would_make_it_wrong(row):
    if row["reason_code"] == "stale_and_low_ctr":
        return (
            "It could be wrong if the page is intentionally evergreen, "
            "its low CTR is normal for the query intent, or the position/CTR data is noisy."
        )
    elif row["reason_code"] == "stale_visible":
        return (
            "It could be wrong if the page is still accurate and useful despite "
            "being old, or if the traffic is not valuable."
        )
    elif row["reason_code"] == "low_ctr_visible":
        return (
            "It could be wrong if the page's search intent naturally produces low CTR "
            "or if the measured position is based on too little stable visibility."
        )
    return (
        "It could be wrong because the rule has little direct evidence "
        "that this page needs intervention."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong, axis=1
)

review_columns = [
    "baseline_rank",
    "content_id",
    "action",
    "reason_code",
    "baseline_action_score",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "confidence_note",
    "what_would_make_it_wrong",
]

top20_review = top20[review_columns]

display(top20_review)

# Save a readable review file locally.
review_path = Path("work/outputs/top20_review.csv")
top20_review.to_csv(review_path, index=False)

print(f"Saved top-20 review to: {review_path}")

,baseline_rank,content_id,action,reason_code,baseline_action_score,impressions_90d,avg_position,ctr,days_since_last_update,confidence_note,what_would_make_it_wrong
0,1,content_fe16a55cd13d,refresh_and_review_ctr,stale_and_low_ctr,1.0,4556,16.4,0.33,194,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
1,2,content_72496874f806,refresh_and_review_ctr,stale_and_low_ctr,1.0,821,5.8,0.24,301,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
2,3,content_6226ee6adc91,refresh_and_review_ctr,stale_and_low_ctr,1.0,545,17.8,0.18,183,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
3,4,content_c2d929d83eaa,refresh_and_review_ctr,stale_and_low_ctr,1.0,7558,17.9,0.20,193,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
4,5,content_cf56e2e2e282,refresh_and_review_ctr,stale_and_low_ctr,1.0,61678,19.7,0.15,194,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
5,6,content_928af3e22c80,refresh_and_review_ctr,stale_and_low_ctr,1.0,1697,15.8,0.12,193,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
6,7,content_0a91db491d14,refresh_and_review_ctr,stale_and_low_ctr,1.0,13299,10.5,0.49,193,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
7,8,content_e3ff1b093148,refresh_and_review_ctr,stale_and_low_ctr,1.0,1408,7.8,0.28,183,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
8,9,content_77d4d5930e5e,refresh_and_review_ctr,stale_and_low_ctr,1.0,828,18.6,0.24,194,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...
9,10,content_7f116ae1f6f5,refresh_and_review_ctr,stale_and_low_ctr,1.0,954,9.0,0.42,301,Moderate: both baseline signals are present.,It could be wrong if the page is intentionally...


Saved top-20 review to: work/outputs/top20_review.csv


## Weak picks + leakage check

The baseline should not be treated as automatically correct.

I will identify at least one weak pick from the top 20 and explain why it deserves human review.

For leakage, I will explicitly check that `trend_pct`, `trend_direction`, and `is_declining_label` are not present in the scoring inputs.

The decline label is useful for evaluating the baseline after ranking, but it is not allowed to influence the ranking itself.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. Weak picks + leakage check

# Find likely weak picks:
# pages with a high score but relatively weak visibility evidence.
weak_candidates = top20[
    (top20["impressions_90d"] < 1000) |
    (top20["avg_position"] == 0) |
    (top20["ctr"].isna())
].copy()

print("Potential weak picks:")
if len(weak_candidates):
    display(
        weak_candidates[
            [
                "baseline_rank",
                "content_id",
                "action",
                "reason_code",
                "impressions_90d",
                "avg_position",
                "ctr",
                "days_since_last_update",
            ]
        ]
    )
else:
    print("No obvious weak pick matched the automatic screen.")
    print(
        "Manual review is still required: inspect the top 20 and identify "
        "the least convincing recommendation."
    )

# ---------- Leakage check ----------
forbidden_inputs = {
    "trend_pct",
    "trend_direction",
    "is_declining_label"
}

score_inputs = {
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
}

leaked = forbidden_inputs.intersection(score_inputs)

print("\nLeakage check:")
print("Forbidden label-derived inputs used in score:", leaked)
assert not leaked, "Leakage detected!"

print("PASS — ranking score uses only current observable signals.")
print("PASS — trend_pct is not used as a feature.")
print("PASS — trend_direction is not used as a feature.")
print("PASS — is_declining_label is not used as a feature.")

# Show the formula again for auditability.
print("\nBaseline formula:")
print("score = 0.60 * stale_visible + 0.40 * low_ctr_visible")

Potential weak picks:


,baseline_rank,content_id,action,reason_code,impressions_90d,avg_position,ctr,days_since_last_update
1,2,content_72496874f806,refresh_and_review_ctr,stale_and_low_ctr,821,5.8,0.24,301
2,3,content_6226ee6adc91,refresh_and_review_ctr,stale_and_low_ctr,545,17.8,0.18,183
8,9,content_77d4d5930e5e,refresh_and_review_ctr,stale_and_low_ctr,828,18.6,0.24,194
9,10,content_7f116ae1f6f5,refresh_and_review_ctr,stale_and_low_ctr,954,9.0,0.42,301
11,12,content_074ba6ead17b,refresh,stale_visible,533,48.0,0.00,183



Leakage check:
Forbidden label-derived inputs used in score: set()
PASS — ranking score uses only current observable signals.
PASS — trend_pct is not used as a feature.
PASS — trend_direction is not used as a feature.
PASS — is_declining_label is not used as a feature.

Baseline formula:
score = 0.60 * stale_visible + 0.40 * low_ctr_visible


## Self-check

- [x] Every section is filled with markdown thinking and supporting code.
- [x] The rule is transparent and uses hand-written thresholds.
- [x] Two observable signals were checked with bucket tables and n.
- [x] At least one signal is linked to the real FlyRank refresh logic: staleness.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Every ranked item has one reason code and one action.
- [x] The top 20 have an action, reason code, confidence note, and what would make the pick wrong.
- [x] Weak picks are explicitly inspected.
- [x] `trend_pct`, `trend_direction`, and `is_declining_label` are not used as ranking inputs.
- [x] IDs are used only for identification/review, not as features.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.